# Deployment e Sistema Multiagente Generativo (`IA3_Projeto`)

Este notebook invoca **todos os modelos treinados** guardados no diretório `models/` (os classificadores `scikit-learn`, o modelo Deep Learning `TensorFlow/Keras` e o agente generativo guardado em `models/generative_agent/`) e executa as duas implementações do sistema multiagente:
1. **Sistema Multiagente com Hugging Face (`transformers`)** — `src/hf_multiagent.py`
2. **Sistema Multiagente com `CrewAI`** — `src/crewai_multiagent.py`

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# Montar Google Drive automaticamente se estiver no Google Colab
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    for pasta in ("/content/drive/MyDrive/Projeto_Futebol", "/content/drive/MyDrive/IA3_Projeto"):
        if Path(pasta).exists():
            os.chdir(pasta)
            break

# Adicionar a raiz do projeto ao sys.path para importar os módulos de src/
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("Diretório raiz do projeto:", ROOT_DIR)

from src.hf_multiagent import (
    GestorModelos,
    SistemaMultiagenteHF,
    preparar_agente_generativo,
)

## 1. Carregar Todos os Classificadores Treinados (`models/`)
Invocamos o `GestorModelos` (definido em `src/hf_multiagent.py`) para carregar todos os ficheiros `.joblib` e `.keras` presentes em `models/`, ler o `models_manifest.json` e extrair uma amostra real do dataset preparado (`data/processed/dataset_futebol_preparado.csv`).

In [ ]:
gestor = GestorModelos()
modelos = gestor.carregar_todos_classificadores()

print(f"Total de classificadores carregados: {len(modelos)}")
for nome, framework in gestor.tipos_modelos.items():
    print(f" - {nome} [{framework}]")

print("\nMelhor modelo identificado:", gestor.obter_nome_melhor_modelo())

In [ ]:
# Extrair uma amostra de teste do dataset preparado e comparar todos os classificadores
amostra_df = gestor.obter_amostra_dataset(indice=-1)
resultados_todos = gestor.prever_todos_modelos(amostra_df)

df_comparacao = pd.DataFrame(resultados_todos)
display(df_comparacao)

## 2. Sistema Multiagente — Versão 1: Hugging Face (`transformers`)
Primeiro garantimos que o modelo generativo e a respetiva configuração estão guardados em `models/generative_agent/`. De seguida, o **Agente Analista** consulta o melhor classificador e o **Agente Relator** redige o relatório técnico sobre a previsão (`FTR`: `H`, `D`, `A`).

In [ ]:
# Descarrega (na primeira execução) e guarda o LLM localmente em models/generative_agent/
caminho_agente = preparar_agente_generativo()
print(f"Agente generativo guardado em: {caminho_agente}")

sistema_hf = SistemaMultiagenteHF(gestor=gestor)
resultado_hf = sistema_hf.executar(amostra_df)

print("=== RELATÓRIO MULTIAGENTE (HUGGING FACE) ===\n")
print(resultado_hf["relatorio"])

## 3. Sistema Multiagente — Versão 2: `CrewAI`
Esta secção executa a versão equivalente do sistema multiagente construída com **CrewAI**, onde os classificadores guardados em `models/` são expostos como ferramentas (`@tool`) invocadas autonomamente pelos agentes.

In [ ]:
# Instalar crewai no Colab se ainda não estiver instalado:
# !pip install -q crewai
from src.crewai_multiagent import executar_crewai

# Nota: Podes usar um modelo local Ollama ('ollama/qwen2.5:3b') ou Hugging Face API
# ('huggingface/Qwen/Qwen2.5-3B-Instruct' com os.environ['HF_TOKEN'] = 'teu_token').
relatorio_crewai = executar_crewai(
    amostra_df,
    gestor=gestor,
    model_uri="huggingface/Qwen/Qwen2.5-3B-Instruct"
)

print("=== RELATÓRIO MULTIAGENTE (CREWAI) ===\n")
print(relatorio_crewai)